In [1]:
# Install a compatible numpy version first to avoid conflicts
!pip install numpy==1.23.5 --force-reinstall

# Then install brightway2 and bw2io, ensuring their dependencies are reinstalled against the correct numpy
# Ensure bw2io is at least version 0.9 for Exiobase3HybridImporter
!pip install brightway2 bw2io>=0.9.0 --force-reinstall

import brightway2 as bw
import bw2io as bi

# 1. Create and switch to a dedicated Brightway project
project_name = "Tequila_LCA_Mexico"
bw.projects.set_current(project_name)

# 2. Setup the default biosphere and LCIA methodologies
if "biosphere3" not in bw.databases:
    bw.bw2setup()

# 3. Import EXIOBASE (Assuming EXIOBASE 3.3 Hybrid SUT datasets)
# For automation, the 'bamboo-lca' library can streamline hybrid matrix generation
db_name = "EXIOBASE_3"
if db_name not in bw.databases:
    print("Importing background database...")
    # Provide the path to your downloaded raw EXIOBASE files
    exio_path = "/path/to/raw/exiobase/data"

    # Using standard brightway2-io iterative importer
    exio_importer = bi.importers.Exiobase3HybridImporter(exio_path, db_name)
    exio_importer.apply_strategies()
    exio_importer.write_database()
    print("EXIOBASE imported successfully!")
else:
    print("EXIOBASE already initialized.")

  Using cached numpy-1.23.5.tar.gz (10.7 MB)
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.
Importing background database...


AttributeError: module 'bw2io.importers' has no attribute 'Exiobase3HybridImporter'

In [ ]:
# Initialize foreground database
fg_db = bw.Database("Tequila_Foreground")
fg_db.register()

# Helper queries to look up EXIOBASE background activities for Mexico (MX) or global markets
exio = bw.Database("EXIOBASE_3")
biosphere = bw.Database("biosphere3")

# Background activity handles (replace strings with precise lookup matches in your environment)
electricity_mx = exio.search("Production of electricity by coal Mexico")[0]
fuel_oil = exio.search("Production of fuel oil")[0]
water_supply = exio.search("Collection, purification and distribution of water")[0]
glass_bottle = exio.search("Manufacture of glass")[0]
co2_fossil = biosphere.search("Carbon dioxide, fossil")[0]

# Define the primary functional unit activity
tequila_bottle = fg_db.new_activity(
    code="reposado_700ml",
    name="100% Reposado Tequila Bottle (700ml, 6-month aged)",
    unit="unit",
    location="MX"
)
tequila_bottle.save()

# Populate the activity exchanges explicitly matching Paper Tables 1 & 2
exchanges = [
    # Production output
    {"input": tequila_bottle.key, "amount": 1.0, "type": "production"},

    # 1. Agave Reception & Clipping Phase
    {"input": exio.search("Cultivation of crops Mexico")[0].key, "amount": 8.62, "type": "technosphere"},  # Agave pineapple
    {"input": electricity_mx.key, "amount": 3.29e-03, "type": "technosphere"},  # Electricity

    # 2. Cooking Phase
    {"input": fuel_oil.key, "amount": 8.06e-01, "type": "technosphere"},
    {"input": water_supply.key, "amount": 7.16e-01, "type": "technosphere"},
    {"input": electricity_mx.key, "amount": 1.19e-04, "type": "technosphere"},

    # 3. Grinding Phase
    {"input": electricity_mx.key, "amount": 1.01e-01, "type": "technosphere"},

    # 4. Fermentation Phase
    {"input": electricity_mx.key, "amount": 9.36e-02, "type": "technosphere"},
    {"input": exio.search("Manufacture of food products")[0].key, "amount": 4.44e-03, "type": "technosphere"},  # Yeast
    {"input": co2_fossil.key, "amount": 3.17e-02, "type": "biosphere"},  # Direct CO2 emissions

    # 5. Distillation 1 & 2
    {"input": fuel_oil.key, "amount": 2.12e-03 + 1.33e-01, "type": "technosphere"},  # Fuel oil combined
    {"input": electricity_mx.key, "amount": 1.96e-01 + 8.08e-01, "type": "technosphere"},  # Electricity combined

    # 6. Post-Distillation Filtering, Rectification & Aging
    {"input": water_supply.key, "amount": 1.97e-01, "type": "technosphere"},
    {"input": electricity_mx.key, "amount": 5.24e-04 + 7.21e-04 + 3.50e-04, "type": "technosphere"},
    {"input": exio.search("Manufacture of chemicals")[0].key, "amount": 6.35e-06 + 1.13e-05, "type": "technosphere"},  # Carbon filters

    # 7. Bottling and Packaging Phase
    {"input": electricity_mx.key, "amount": 5.07e-04, "type": "technosphere"},
    {"input": glass_bottle.key, "amount": 5.50e-01, "type": "technosphere"},  # 550g Glass bottle
    {"input": exio.search("Manufacture of aluminum")[0].key, "amount": 9.80e-02, "type": "technosphere"},  # Aluminum cap
    {"input": exio.search("Manufacture of wood products")[0].key, "amount": 2.45e-01, "type": "technosphere"},  # Wooden box
]

# Write exchanges to the database activity
for exchange in exchanges:
    tequila_bottle.new_exchange(**exchange).save()

print("Inventory created and bound safely.")


In [ ]:
# Define target methods based on your study requirements
cml_gwp = [m for m in bw.methods if "CML" in m[0] and "global warming" in m[1]][0]
recipe_endpoint = [m for m in bw.methods if "ReCiPe" in m[0] and "Endpoint" in m[1]][0]

# Perform calculation
lca = bw.LCA({tequila_bottle: 1}, cml_gwp)
lca.lci()
lca.lcia()

print(f"Replicated Climate Footprint (CML): {lca.score:.4f} kg CO2-eq per bottle")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import brightway2 as bw

# Ensure the active workspace is correct
bw.projects.set_current("Tequila_LCA_Mexico")
tequila_activity = bw.Database("Tequila_Foreground").search("Reposado Tequila")[0]

# --- 1. NUMERICAL OUTPUT: MULTI-METHOD SUMMARY ---
selected_methods = [
    ("CML-IA baseline", "global warming", "GWP100a"),
    ("CML-IA baseline", "abiotic depletion", "fossil fuels"),
    ("CML-IA baseline", "human toxicity", "human toxicity"),
    ("ReCiPe Endpoint (H)", "Total", "Total Endpoint Impact")
]

summary_data = []
for m in selected_methods:
    try:
        target_method = [method for method in bw.methods if m[0] in method[0] and m[1] in method[1]][0]
        lca = bw.LCA({tequila_activity: 1}, target_method)
        lca.lci()
        lca.lcia()
        summary_data.append({
            "Method Family": m[0],
            "Impact Category": m[1],
            "Score": lca.score,
            "Unit": bw.methods[target_method].get('unit', 'unknown')
        })
    except IndexError:
        continue

df_summary = pd.DataFrame(summary_data)
df_summary.to_csv("tequila_lcia_totals.csv", index=False)
print("Saved total impact summary to CSV.")


# --- 2. NUMERICAL & GRAPHICAL OUTPUT: HOTSPOT CONTRIBUTION ---
# Focus heavily on Carbon Footprint (GWP100a)
gwp_method = [m for m in bw.methods if "CML" in m[0] and "global warming" in m[1]][0]
lca_gwp = bw.LCA({tequila_activity: 1}, gwp_method)
lca_gwp.lci()
lca_gwp.lcia()

# Traverse immediate foreground exchanges
contribution_data = []
for exc in tequila_activity.exchanges():
    if exc['type'] == 'production':
        continue
    # Redo calculation isolating this single structural stream
    lca_gwp.redo_lcia({exc.input: exc['amount']})
    contribution_data.append({
        "Stage": exc.input['name'],
        "Absolute GWP (kg CO2-eq)": lca_gwp.score
    })

df_contrib = pd.DataFrame(contribution_data)
# Add percentage calculation
total_gwp_score = df_contrib["Absolute GWP (kg CO2-eq)"].sum()
df_contrib["Percentage Contribution (%)"] = (df_contrib["Absolute GWP (kg CO2-eq)"] / total_gwp_score) * 100
df_contrib = df_contrib.sort_values(by="Absolute GWP (kg CO2-eq)", ascending=False)

# Save Table
df_contrib.to_csv("tequila_gwp_process_contribution.csv", index=False)

# Render Chart
plt.figure(figsize=(10, 5))
sns.barplot(
    data=df_contrib.head(6),  # Plot top 6 hotspots
    y="Stage",
    x="Percentage Contribution (%)",
    palette="viridis"
)
plt.title("Carbon Footprint (GWP100a) Foreground Hotspot Analysis")
plt.xlabel("Share of Total Global Warming Impact (%)")
plt.ylabel("Lifecycle Process Stage")
plt.tight_layout()

# Save Visual
plt.savefig("tequila_hotspot_chart.png", dpi=300)
plt.close()
print("Saved process contribution tables and visualization charts successfully.")
